# Person 4 (Neha): third-set Hinglish features
Computes tokens, words and fertility for the 200 third-set **Hinglish** texts with the router's own tokenizer (same recipe as `features_confirmatory_v1.csv`).
Touches **no scores**. Plan: `docs/correction_model_plan_v1.md` (frozen).
Output: `features_thirdset_v1.csv` (new file, never overwritten).

## 1. Install

In [ ]:
!pip install -q transformers sentencepiece

## 2. Upload 3 files
`thirdset_v2.csv`, `confirmatory_v1.csv`, `features_confirmatory_v1.csv`

In [ ]:
from google.colab import files
up = files.upload()
print(list(up))

## 3. Check the input file is the frozen one

In [ ]:
import hashlib
def sha256(p):
    h = hashlib.sha256()
    with open(p,'rb') as f:
        for ch in iter(lambda: f.read(1<<20), b''): h.update(ch)
    return h.hexdigest()
EXPECTED = '91d82d48a161527174fa82b47a9e041f8ad379ebbf7dc071147f05540b13a826'
got = sha256('thirdset_v2.csv'); print(got)
assert got == EXPECTED, 'thirdset_v2.csv is NOT the frozen file. Stop.'
print('OK: frozen third-set file')

## 4. Load the router's tokenizer and define the recipe

In [ ]:
import unicodedata, pandas as pd
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('routellm/bert_gpt4_augmented')
print(type(tok).__name__, tok.name_or_path)

def compute_features(df, text_col='hinglish'):
    texts = [unicodedata.normalize('NFC', str(x).strip()) for x in df[text_col]]
    f = pd.DataFrame({'pair_id': df['pair_id'].values})
    f[f'{text_col}_words']  = [len(t.split()) for t in texts]
    f[f'{text_col}_tokens'] = [len(tok.tokenize(t)) for t in texts]
    f[f'{text_col}_fertility'] = f[f'{text_col}_tokens'] / f[f'{text_col}_words']
    return f

## 5. PROOF CHECK: reproduce the team's saved confirmatory features
If this fails, do NOT continue and send me the printed output.

In [ ]:
conf = pd.read_csv('confirmatory_v1.csv'); saved = pd.read_csv('features_confirmatory_v1.csv')
mine = compute_features(conf)
m = mine.merge(saved, on='pair_id', suffixes=('_mine','_saved'))
ok_w = (m.hinglish_words_mine == m.hinglish_words_saved).mean()
ok_t = (m.hinglish_tokens_mine == m.hinglish_tokens_saved).mean()
print('words match:', ok_w, '| tokens match:', ok_t, '| rows:', len(m))
assert ok_w == 1.0 and ok_t == 1.0, 'Tokenizer/recipe differs from the team. STOP and ask.'
print('PROOF OK: same recipe and tokenizer as the team')

## 6. Compute the third-set features and save

In [ ]:
import os
third = pd.read_csv('thirdset_v2.csv')
assert len(third) == 200 and third.pair_id.is_unique and third.hinglish.notna().all()
feats = compute_features(third)
assert (feats.hinglish_words > 0).all()
assert not os.path.exists('features_thirdset_v1.csv'), 'file exists: never overwrite'
feats.to_csv('features_thirdset_v1.csv', index=False, lineterminator='\n')
print(feats.describe().round(2))
print('sha256:', sha256('features_thirdset_v1.csv'))
files.download('features_thirdset_v1.csv')